<div style="background: linear-gradient(90deg,#0f172a,#1e293b,#334155);
            padding:25px;
            border-radius:10px;
            color:white;">
            
<h1>🚀 Customer Transactions ETL Pipeline</h1>

<h3>Data Engineering Project</h3>

<p>
Source Systems → Data Ingestion/Extraction → Transformation → Validation → Data Warehouse
</p>

</div>

### Module imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, explode, trim
from pyspark.sql.functions import regexp_replace, col
from pyspark.sql.functions import array_contains
from pyspark.sql.functions import year, month, dayofmonth, to_timestamp
import os
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
spark = SparkSession.builder \
    .appName("ReadCSV") \
    .getOrCreate()

<div style="background: linear-gradient(90deg,#0f172a,#1e293b,#334155);
            padding:25px;
            border-radius:10px;
            color:white;">
            


<h3>Extraction</h3>


</div>

In [3]:
raw_data_set = spark.read.csv(
    r"C:\Users\patsi\Documents\Virtual_data_department\Data_set\Retail_Transactions_Dataset.csv",
    header=True,
    inferSchema=True
)

## Transform

In [4]:
# 3. Split products into array
raw_data_set_split = raw_data_set.withColumn(
    "Product",
    split("Product", ",")
)

# 4. Explode into rows
raw_data_set_exploded = raw_data_set_split.withColumn(
    "Product",
    explode("Product")
)

# 5. Clean whitespace (best practice)
raw_data_set_final = raw_data_set_exploded.withColumn("Product", trim("Product")) \
                      .select("Transaction_ID", "Product")

In [5]:
def keep_alphanumerics_and_spaces(df, column_name, new_column_name=None):
    if new_column_name is None:
        new_column_name = column_name
    return df.withColumn(
        new_column_name,
        regexp_replace(col(column_name), r"[^a-zA-Z0-9 ]", "")
    )

In [6]:
raw_data_set_product_clean = keep_alphanumerics_and_spaces(raw_data_set_final, "Product", "Product_Clean")

In [7]:


result = (
    raw_data_set.alias("transactions")
    .join(
        raw_data_set_product_clean.alias("products"),
        array_contains(
            split(col("transactions.Product"), ","),
            col("products.Product_Clean")
        ),
        "inner"
    )
    .select(
        col("transactions.*"),
        col("products.Product_Clean").alias("Matched_Product")
    )
)

In [21]:
raw_data_set_product_clean = raw_data_set_product_clean.select("Product_Clean").dropDuplicates()

In [10]:
pricing_matrix = spark.read.csv(
    r"C:\Users\patsi\Documents\Virtual_data_department\Data_engineering\Extract\products_pricing_matrix.csv",
    header=True,
    inferSchema=True
)